# XYZ Stage G-code Generator

This notebook generates a G-code file that drives an XYZ motion stage through a grid of points (a raster/sweep pattern) and, at every point, performs a sequence of moves: descend to a working height, trigger a sensor (or tool) for a short period, wait, then retract to a safe height before moving to the next point.

**Typical use case:** sweeping a sensor (e.g. a spectrophotometer probe, nicknamed `VideoBarrelino` in this code) across a flat sample or plate to take one measurement per grid position.

**Workflow:**
1. Run the Sweep functions cells to define the path-generation and G-code-generation functions.
2. Run the Parameters cell to set your grid limits/steps and generate the `.gcode` file.
3. Load the resulting `trajectory.gcode` file into your machine controller (e.g. grblControl, Universal G-code Sender, Mach3) and run it on the XYZ stage.

# Sweep functions

### `zigzag` — boustrophedon (back-and-forth) path generator

Generates a list of `(X, Y, Z)` points that sweep the rectangular area defined by `x_start`→`x_end` and `y_start`→`y_end`, alternating direction on every row (left-to-right, then right-to-left, and so on), this is the classic "boustrophedon" or zigzag raster pattern, which minimizes travel time by avoiding a full retrace to the start of each row.

> **Note:** this function is defined here but is **not** the one used by default in the *Parameters* cell below (which uses `horizontal_path` instead). Use `zigzag` instead of `horizontal_path` in the Parameters cell if you prefer alternating-direction rows over always restarting each row from the same X position.

> **Inputs:** `x_start`/`x_end` (X range), `y_start`/`y_end` (Y range), `step_x`/`step_y` (step size in X and Y), `z` (constant Z height for every point in the list, this is *not* the working or safe height used during the G-code moves, see `generar_gcode` below).
> **Output:** a list of `(x, y, z)` tuples describing the full sweep path, in visiting order.

In [ ]:
def zigzag(x_start, x_end, y_start, y_end, step_x, step_y, z):
    """
    Generates a list of (X, Y, Z) points in a raster sweep pattern.
    The sweep is done line by line, alternating direction (zigzag).
    """
    points = []
    y = y_start
    direction = 1  # 1 = moves in positive X, -1 = negative X

    while y <= y_end:
        if direction == 1:
            xs = list(range(x_start, x_end + step_x, step_x))
        else:
            xs = list(range(x_end, x_start - step_x, -step_x))

        for x in xs:
            points.append((x, y, z))

        y += step_y
        direction *= -1  # reverse direction for the next line

    return points

### `horizontal_path` — same-direction (left-to-right) path generator

Generates a list of `(X, Y, Z)` points that sweep the same rectangular area as `zigzag`, but every row is always traversed in the same direction (from `x_start` to `x_end`). After finishing a row, the stage jumps back to `x_start` at the next Y level before continuing, simpler to reason about than the zigzag pattern, at the cost of an extra retrace move per row.

This is the function actually used by the Parameters cell below to build the default sweep.

> **Inputs:** same as `zigzag` (`x_start`/`x_end`, `y_start`/`y_end`, `step_x`/`step_y`, `z`).
> **Output:** a list of `(x, y, z)` tuples, with consecutive duplicate points removed for safety.

In [ ]:
def horizontal_path(x_start, x_end, y_start, y_end, step_x, step_y, z):
    points = []
    y = y_start

    while y <= y_end:
        # Move along X from start to end
        for x in range(x_start, x_end + step_x, step_x):
            if x <= x_end:
                points.append((x, y, z))

        # Move to the next line
        y += step_y

        # If we haven't finished yet, move up in Y and add only the starting point
        if y <= y_end:
            points.append((x_start, y, z))

    # Remove consecutive duplicates (for safety)
    clean_points = [points[0]]
    for p in points[1:]:
        if p != clean_points[-1]:
            clean_points.append(p)

    return clean_points

### `generate_gcode` — turns a point list into a runnable G-code program

Takes the list of `(X, Y, Z)` points produced above and writes a full G-code program that, for each point, performs:

1. Moves to the point's X/Y position while staying at the safe height (`z_safe`), avoids dragging the tool/sensor across the sample while traveling between points.
2. Descends to the working height (`z_work`) at the given `feedrate`.
3. Activates the sensor (`M8`, labeled `Barrelino` in this setup) and waits `wait_time_1` seconds, this is the actual measurement/exposure window.
4. Deactivates the sensor (`M9`).
5. Waits an additional `wait_time_2` seconds (e.g. to let the controller/PC finish logging the reading before the stage moves again).
6. Retracts back to the safe height before moving to the next point.

At the end of the program, the stage returns to the origin (`X0 Y0`) at the safe height and the program halts (`M30`).

> **Inputs to tune for your setup:**
> - `feedrate` — XY/Z travel speed in mm/min for the `G1` (working) moves.
> - `wait_time_1` — how long the sensor stays active at each point (seconds). Set this to match your sensor's integration/measurement time.
> - `wait_time_2` — extra settling/logging delay after deactivating the sensor (seconds).
> - `z_safe` — the retract height used for all XY travel moves (should clear any obstacles/the sample itself).
> - `z_work` — the height at which the sensor actually takes a reading (typically just touching or just above the sample surface).
> **Output:** a single string containing the complete G-code program, ready to be written to a `.gcode` file.

In [ ]:
def generate_gcode(points, feedrate=500, wait_time_1=0.5, wait_time_2=5, z_safe=-3, z_work=0):
    """
    Generates a G-code program that travels through a list of (X, Y, Z) points, performing at each one:
    - Move to a safe position
    - Descend to the working Z height
    - Activate/deactivate the sensor
    - Return to safe height
    """
    gcode = []
    gcode.append("; --- Start of G-code program ---")
    gcode.append("G21 ; Units in millimeters")
    gcode.append("G90 ; Absolute coordinates")
    gcode.append("G0 Z{:.3f} ; Move down to initial safe height".format(z_safe))
    gcode.append("")

    for i, (x, y, z) in enumerate(points):
        gcode.append(f"; --- Move {i+1} to X{x} Y{y} ---")

        # 1. Move over the point at safe height
        gcode.append(f"G0 X{x:.3f} Y{y:.3f} Z{z_safe:.3f}")

        # 2. Descend to the working plane
        gcode.append(f"G1 Z{z_work:.3f} F{feedrate}")

        # 3. Activate sensor
        gcode.append("M8 ; Activate Barrelino")
        gcode.append(f"G4 P{wait_time_1} ; Wait {wait_time_1}s")

        # 4. Deactivate sensor
        gcode.append("M9 ; Deactivate Barrelino")

        # 5. Wait before the next move
        gcode.append(f"G4 P{wait_time_2} ; Wait {wait_time_2}s")

        # 6. Retract to safe height
        gcode.append(f"G0 Z{z_safe:.3f}\n")


    gcode.append("; --- End of path ---")
    gcode.append("G0 X0 Y0 Z{:.3f} ; Return to safe origin".format(z_safe))
    gcode.append("M30 ; End of program")
    gcode.append("; --- End of G-code ---")

    return "\n".join(gcode)

# Parameters

### Set your grid and timing, then generate the `.gcode` file

This is the only cell you normally need to edit. It defines the physical sweep area and timing, builds the point list with `horizontal_path`, generates the G-code with `generate_gcode`, and writes it to `trajectory.gcode` in the current working directory.

> **What to change:**
> - `x_start` / `x_end` / `step_x` — X range (mm) and step size of the grid.
> - `y_start` / `y_end` / `step_y` — Y range (mm) and step size of the grid.
> - `z` — constant Z value attached to every generated point (passed through to `horizontal_path`; it is **not** the safe/working height used during moves — those come from `z_safe`/`z_work` below).
> - `feedrate` — travel speed (mm/min) for the working (`G1`) moves.
> - `wait_time_1` — sensor active/measurement time (seconds) at each point.
> - `wait_time_2` — extra wait after the sensor turns off (seconds), before retracting.
> - `z_safe` — safe retract height (mm) used for all travel moves between points.
> - `z_work` — working height (mm) where the sensor takes its reading.
> - Swap `horizontal_path(...)` for `zigzag(...)` if you want alternating-direction rows instead of always restarting each row from `x_start`.

After running this cell, you should see `Generated G-code file` printed, and a new `trajectory.gcode` file will appear next to this notebook, load it into your XYZ stage controller to run the sweep.

In [ ]:
x_start = 0
x_end = 40
step_x = 5
y_start = 0
y_end = 30
step_y = 5
z = 0

points = horizontal_path(x_start, x_end, y_start, y_end, step_x, step_y, z)

gcode_text = generate_gcode(points, feedrate=500, wait_time_1=0.5, wait_time_2=50, z_safe=-3, z_work=0)

with open("trajectory.gcode", "w") as f:
    f.write(gcode_text)

print("Generated G-code file")
